###### Imports and Settings

In [1]:
import pandas as pd
#import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
from functools import reduce
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

###### Functions

In [2]:
#function for percent of whole
def percent(x, y):
    try:
        return round((x/y)*100, 2)
    except ZeroDivisionError:
        return 666
#function for population density
def populationdensity(x, y):
    try:
        return round(x/y, 2)
    except ZeroDivisionError:
        return 666

# ACS 2010

In [3]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [4]:
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']
#api_key = '24fc7d81b74510d599f702dbd408fb18e1466d81'

In [5]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149', #Rutherford
       '119'] #Maury

## Places Needed:  
### Per County:  
+ Cheatham County: Ashland City town, Kingston Springs town, Pegram town, Pleasant View city  
+ Davidson County: Belle Meade city, Berry Hill city, Forest Hills city, **Goodlettsville** city, 
Nashville-Davidson metropolitan government (balance), Oak Hill city, **Ridgetop** city  
+ Dickson County: Burns town, Charlotte town, Dickson city, Slayden town, Vanleer town, White Bluff town  
+ Houston County: Erin city, **Tennessee Ridge** town  
+ Humphreys County: McEwen city, New Johnsonville city, Waverly city  
+ Maury County: Columbia city, Mount Pleasant city, **Spring Hill** city  
+ Montgomery County: Clarksville city  
+ Robertson County: Adams city, Cedar Hill city, Coopertown town, Cross Plains city, Greenbrier town, **Millersville** city, **Portland** city, **Ridgetop** city, Springfield city, **White House** city  
+ Rutherford County: Eagleville city, La Vergne city, Murfreesboro city, Smyrna town
+ Stewart County: Cumberland City town, Dover city, **Tennessee Ridge** town  
+ Sumner County: Gallatin city, **Goodlettsville** city, Hendersonville city, **Millersville** city, **Portland** city, Westmoreland town, **White House** city  
+ Trousdale County: Hartsville  
+ Williamson County: Brentwood city, Fairview city, Franklin city, Nolensville town, **Spring Hill** city, Thompson's Station town  
+ Wilson County: Lebanon city, Mount Juliet city, Watertown city  

So the ones without overlapping places are: Cheatham, Dickson, Humphreys, Montgomery, Rutherford, Trousdale, and Wilson Counties  

In [6]:
qlaces = ['1600000US4702180', #Ashland City: Cheatham
          '1600000US4739660', #Kingston Springs: Cheatham
          '1600000US4757480', #Pegram: Cheatham
          '1600000US4759560', #Pleasant View: Cheatham
          '1600000US4704620', #Belle Meade: Davidson
          '1600000US4705140', #Berry Hill: Davidson
          '1600000US4727020', #Forest Hills: Davidson
          '1600000US4729920', #Goodlettsville: Davidson/Sumner
          '1600000US4752006', #Nashville-Davidson metropolitan government (balance): Davidson
          '1600000US4754780', #Oak Hill: Davidson
          '1600000US4763140', #Ridgetop: Davidson/Robertson
          '1600000US4709880', #Burns: Dickson
          '1600000US4713080', #Charlotte: Dickson
          '1600000US4720620', #Dickson: Dickson
          '1600000US4769080', #Slayden: Dickson
          '1600000US4776860', #Vanleer: Dickson
          '1600000US4779980', #White Bluff: Dickson
          '1600000US4724320', #Erin: Houston
          '1600000US4773460', #Tennessee Ridge: Houston/Stewart
          '1600000US4744840', #McEwen: Humphreys
          '1600000US4752820', #New Johnsonville: Humphreys
          '1600000US4778560', #Waverly: Humphreys
          '1600000US4716540', #Columbia: Maury
          '1600000US4751080', #Mount Pleasant: Maury
          '1600000US4770580', #Spring Hill: Maury/Williamson
          '1600000US4715160', #Clarksville: Montgomery
          '1600000US4700200', #Adams: Robertson
          '1600000US4711980', #Cedar Hill: Robertson
          '1600000US4716980', #Coopertown: Robertson
          '1600000US4718420', #Cross Plains: Robertson
          '1600000US4730960', #Greenbrier: Robertson
          '1600000US4748980', #Millersville: Robertson/Sumner
          '1600000US4760280', #Portland: Robertson/Sumner
          '1600000US4770500', #Springfield: Robertson
          '1600000US4780200', #White House: Robertson/Sumner
          '1600000US4722360', #Eagleville: Rutherford
          '1600000US4741200', #La Vergne: Rutherford
          '1600000US4751560', #Murfreesboro: Rutherford
          '1600000US4769420', #Smyrna: Rutherford
          '1600000US4718820', #Cumberland City: Stewart
          '1600000US4721400', #Dover: Stewart
          '1600000US4728540', #Gallatin: Sumner
          '1600000US4733280', #Hendersonville: Sumner
          '1600000US4779420', #Westmoreland: Sumner
          '1600000US4708280', #Brentwood: Williamson
          '1600000US4725440', #Fairview: Williamson
          '1600000US4727740', #Franklin: Williamson
          '1600000US4753460', #Nolensville: Williamson
          '1600000US4773900', #Thompson's Station: Williamson
          '1600000US4741520', #Lebanon: Wilson
          '1600000US4750780', #Mount Juliet: Wilson
          '1600000US4778320'] #Watertown: Wilson


## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
+ dg3: ID's 93-138  
+ dg4: ID's 139-184  
+ dg5: ID's 185-230  
+ dg6: ID's 231-276  
+ dg7: ID's 277-322  
+ dg8: ID's 323-368  
+ dg9: ID's 369-414  
+ dg10: ID's 415-460  
+ dg11: ID's 461-506  
+ dg12: ID's 507-552  
+ dg13: ID's 553-598  

**This data guide stops at ID 554 as of 7/8/2022.**

In [7]:
dataguide = pd.read_csv('../data guides/DATA GUIDE ACS 2010.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [8]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]
dg3 = dataguide[dataguide['ID'].between(93, 138)]
dg4 = dataguide[dataguide['ID'].between(139, 184)]
dg5 = dataguide[dataguide['ID'].between(185, 230)]
dg6 = dataguide[dataguide['ID'].between(231, 276)]
dg7 = dataguide[dataguide['ID'].between(277, 322)]
dg8 = dataguide[dataguide['ID'].between(323, 368)]
dg9 = dataguide[dataguide['ID'].between(369, 414)]
dg10 = dataguide[dataguide['ID'].between(415, 460)]
dg11 = dataguide[dataguide['ID'].between(461, 506)]
dg12 = dataguide[dataguide['ID'].between(507, 552)]
dg13 = dataguide[dataguide['ID'].between(553, 598)]

## One Through Twenty Seven

In [9]:
# ONE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg1['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2010/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
one = data_appended
#places call
url_str= 'https://api.census.gov/data/2010/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
one = one.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2010/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
one = one.append(state, ignore_index = True)
onepull = one
print('Okay Finished')

Okay Finished


In [10]:
#placemarker
one = onepull

In [11]:
# TWO
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg2['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg2['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2010/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
two = data_appended
#places call
url_str= 'https://api.census.gov/data/2010/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
two = two.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2010/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
two = two.append(state, ignore_index = True)
twopull = two
print('Okay Finished')

Okay Finished


In [12]:
two = twopull

In [14]:
# THREE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg3['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg3['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
three = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
three = three.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
three = three.append(state, ignore_index = True)
threepull = three
print('Okay Finished')

Okay Finished


In [15]:
three = threepull

In [16]:
# FOUR
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg4['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg4['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
four = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
four = four.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
four = four.append(state, ignore_index = True)
fourpull = four
print('Okay Finished')

Okay Finished


In [17]:
four = fourpull

In [18]:
# FIVE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg5['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg5['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
five = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
five = five.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
five = five.append(state, ignore_index = True)
fivepull = five
print('Okay Finished')

Okay Finished


In [19]:
five = fivepull

In [20]:
# SIX
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg6['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg6['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
six = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
six = six.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
six = six.append(state, ignore_index = True)
sixpull = six
print('Okay Finished')

Okay Finished


In [21]:
six = sixpull

In [22]:
# SEVEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg7['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg7['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
seven = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
seven = seven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
seven = seven.append(state, ignore_index = True)
sevenpull = seven
print('Okay Finished')

Okay Finished


In [23]:
seven = sevenpull

In [24]:
# EIGHT
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg8['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg8['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eight = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
eight = eight.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eight = eight.append(state, ignore_index = True)
eightpull = eight
print('Okay Finished')

Okay Finished


In [25]:
eight = eightpull

In [26]:
# NINE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg9['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg9['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
nine = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
nine = nine.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
nine = nine.append(state, ignore_index = True)
ninepull = nine
print('Okay Finished')

Okay Finished


In [27]:
nine = ninepull

In [28]:
# TEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg10['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg10['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
ten = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
ten = ten.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
ten = ten.append(state, ignore_index = True)
tenpull = ten
print('Okay Finished')

Okay Finished


In [29]:
ten = tenpull

In [30]:
# ELEVEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg11['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg11['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eleven = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
eleven = eleven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eleven = eleven.append(state, ignore_index = True)
elevenpull = eleven
print('Okay Finished')

Okay Finished


In [31]:
eleven = elevenpull

In [32]:
# Twelve
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg12['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg12['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twelve = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
twelve = twelve.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twelve = twelve.append(state, ignore_index = True)
twelvepull = twelve
print('Okay Finished')

Okay Finished


In [33]:
twelve = twelvepull

In [34]:
# Thirteen
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg13['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg13['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
thirteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
thirteen = thirteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
thirteen = thirteen.append(state, ignore_index = True)
thirteenpull = thirteen
print('Okay Finished')

Okay Finished


In [35]:
thirteen = thirteenpull

In [37]:
last = thirteen

In [39]:
dfs = [two,three,four,five,six,seven,eight,nine,ten,eleven,twelve]

In [40]:
for df in dfs:
    df = df.drop(columns = ['NAME','StateFIPS','GeoFIPS'], inplace = True)

In [41]:
one = one.drop(columns = ['StateFIPS','GeoFIPS'])
last = last.drop(columns = 'NAME')

The default inplace=False is intended for assigning back to the original dataframe, because it returns a new copy. However inplace=True operates on the same copy and returns None, therefore there is no need to assign back to the original dataframe.

In [42]:
df_merged = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)

In [43]:
dfs = [one, df_merged, last]
data = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)

In [45]:
transp = data.transpose()
transp.columns = transp.iloc[0]

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Adams city, Tennessee","Ashland City town, Tennessee","Belle Meade city, Tennessee","Berry Hill city, Tennessee","Brentwood city, Tennessee","Burns town, Tennessee","Cedar Hill city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Columbia city, Tennessee","Coopertown town, Tennessee","Cross Plains city, Tennessee","Cumberland City town, Tennessee","Dickson city, Tennessee","Dover city, Tennessee","Eagleville city, Tennessee","Erin city, Tennessee","Fairview city, Tennessee","Forest Hills city, Tennessee","Franklin city, Tennessee","Gallatin city, Tennessee","Goodlettsville city, Tennessee","Greenbrier town, Tennessee","Hendersonville city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Millersville city, Tennessee","Mount Juliet city, Tennessee","Mount Pleasant city, Tennessee","Murfreesboro city, Tennessee","Nashville-Davidson metropolitan government (balance), Tennessee","New Johnsonville city, Tennessee","Nolensville town, Tennessee","Oak Hill city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Portland city, Tennessee","Ridgetop city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Springfield city, Tennessee","Spring Hill city, Tennessee","Tennessee Ridge town, Tennessee","Thompson's Station town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","Westmoreland town, Tennessee","White Bluff town, Tennessee","White House city, Tennessee",Tennessee
NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Adams city, Tennessee","Ashland City town, Tennessee","Belle Meade city, Tennessee","Berry Hill city, Tennessee","Brentwood city, Tennessee","Burns town, Tennessee","Cedar Hill city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Columbia city, Tennessee","Coopertown town, Tennessee","Cross Plains city, Tennessee","Cumberland City town, Tennessee","Dickson city, Tennessee","Dover city, Tennessee","Eagleville city, Tennessee","Erin city, Tennessee","Fairview city, Tennessee","Forest Hills city, Tennessee","Franklin city, Tennessee","Gallatin city, Tennessee","Goodlettsville city, Tennessee","Greenbrier town, Tennessee","Hendersonville city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Millersville city, Tennessee","Mount Juliet city, Tennessee","Mount Pleasant city, Tennessee","Murfreesboro city, Tennessee",Nashville-Davidson metropolitan government (ba...,"New Johnsonville city, Tennessee","Nolensville town, Tennessee","Oak Hill city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Portland city, Tennessee","Ridgetop city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Springfield city, Tennessee","Spring Hill city, Tennessee","Tennessee Ridge town, Tennessee","Thompson's Station town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","Westmoreland town, Tennessee","White Bluff town, Tennessee","White House city, Tennessee",Tennessee
GEO_ID,0500000US47161,0500000US47125,0500000US47083,0500000US47085,0500000US47043,0500

In [46]:
transp = transp.loc[transp['Stewart County, Tennessee'] != 'Stewart County, Tennessee']
transp = transp.loc[transp['Stewart County, Tennessee'] != '0500000US47161']

In [47]:
numcols = list(transp.columns)
numcols
transp[numcols] = transp[numcols].astype(float)

In [48]:
data = transp

In [49]:
GNRCCounties = [data['Stewart County, Tennessee'],data['Montgomery County, Tennessee'],data['Houston County, Tennessee'],data['Humphreys County, Tennessee'],
                data['Dickson County, Tennessee'],data['Cheatham County, Tennessee'],data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],
                data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],data['Trousdale County, Tennessee'],data['Williamson County, Tennessee'],
                data['Rutherford County, Tennessee']]
data['GNRC'] = sum(GNRCCounties)

In [50]:
MPOCounties = [data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],
               data['Williamson County, Tennessee'],data['Rutherford County, Tennessee'],data['Maury County, Tennessee']]
data['MPO'] = sum(MPOCounties)

In [53]:
RuthInc = [data['Eagleville city, Tennessee'],data['La Vergne city, Tennessee'],data['Murfreesboro city, Tennessee'],data['Smyrna town, Tennessee']]
data['Rutherford Incorporated'] = sum(RuthInc)
data['Rutherford Unincorporated'] = data['Rutherford County, Tennessee'] - data['Rutherford Incorporated']
WilsonInc = [data['Lebanon city, Tennessee'],data['Mount Juliet city, Tennessee'],data['Watertown city, Tennessee']]
data['Wilson Incorporated'] = sum(WilsonInc)
data['Wilson Unincorporated'] = data['Wilson County, Tennessee'] - data['Wilson Incorporated']
CheathInc = [data['Ashland City town, Tennessee'],data['Kingston Springs town, Tennessee'],data['Pegram town, Tennessee'],data['Pleasant View city, Tennessee']]
data['Cheatham Incorporated'] = sum(CheathInc)
data['Cheatham Unincorporated'] = data['Cheatham County, Tennessee'] - data['Cheatham Incorporated']
DicksInc = [data['Burns town, Tennessee'],data['Charlotte town, Tennessee'],data['Dickson city, Tennessee'],data['Slayden town, Tennessee'],
            data['Vanleer town, Tennessee'],data['White Bluff town, Tennessee']]
data['Dickson Incorporated'] = sum(DicksInc)
data['Dickson Unincorporated'] = data['Dickson County, Tennessee'] - data['Dickson Incorporated']
HumphInc = [data['McEwen city, Tennessee'],data['New Johnsonville city, Tennessee'],data['Waverly city, Tennessee']]
data['Humphreys Incorporated'] = sum(HumphInc)
data['Humphreys Unincorporated'] = data['Humphreys County, Tennessee'] - data['Humphreys Incorporated']
data['Montgomery Incorporated'] = data['Clarksville city, Tennessee']
data['Montgomery Unincorporated'] = data['Montgomery County, Tennessee'] - data['Montgomery Incorporated']
#data['Trousdale Incorporated'] = data['Hartsville/Trousdale County, Tennessee']
#data['Trousdale Unincorporated'] = data['Trousdale County, Tennessee'] - data['Trousdale Incorporated']

Not dropping any places right now.

In [54]:
ok = data.transpose().reset_index()

In [55]:
ok.head()

,NAME,agebysex_total_series,hhsize_avg,poverty_total_bysexbyage_series,poverty_belowlevel,units_allhousing,occupancy_total_series,occupancy_occupiedunits,occupancy_vacantunits,tenure_total_series,tenure_owneroccunits,tenure_renteroccunits,units_total_series,units_one_detached,units_one_attached,units_two,units_threetofour,units_fivetonine,units_tentonineteen,units_twentytofortynine,units_fiftyormore,units_mobilehome,units_boatrvvanetc,hhtype_total_series,hhtype_familyhh,hhtype_familyhh_marriedcouplefam,hhtype_familyhh_otherfam,hhtype_familyhh_malenospouse,hhtype_familyhh_femalenospouse,hhtype_nonfamhh,hhtype_nonfamhh_householderalone,hhtype_nonfamhh_householdernotalone,hhincome_median,hhincome_total_series,hhincome_lessthan10000,hhincome_10to14999,hhincome_15to19999,hhincome_20to24999,hhincome_25to29999,hhincome_30to34999,hhincome_35to39999,hhincome_40to44999,hhincome_45to49999,hhincome_50to59999,hhincome_60to74999,hhincome_75to99999,hhincome_100to124999,hhincome125to149999,hhincome150to199999,hhincome200ormore,housingcost_total_selectedownercosts%hhincome_series,housingcost_total%ownercostwmortgage_series,housingcost_%ownercost30to34.9_wmortgage,housingcost_%ownercost35to39.9_wmortgage,housingcost_%ownercost40to49.9_wmortgage,housingcost_%ownercost50+_wmortgage,housingcost_total%ownercostwomortgage_series,housingcost_%ownercost30to34.9_womortgage,housingcost_%ownercost35to39.9_womortgage,housingcost_%ownercost40to49.9_womortgage,housingcost_%ownercost50+_womortgage,housingcost_total_rent%hhincome_series,housingcost_%rentercost30to34.9,housingcost_%rentercost35to39.9,housingcost_%rentercost40to49.9,housingcost_%rentercost50+,housingcost_medvalue_ownerocc_wmortgage,housingcost_medvalue_ownerocc_womortgage,housingcost_mediangrossrent_renteroccupied,commute_total_meansoftransportationtowork_series,commute_cartruckvan,commute_cartruckvan_drovealone,commute_cartruckvan_carpooled,commute_cartruckvan_carpooled_2ppl,commute_cartruckvan_carpooled_3ppl,commute_cartruckvan_carpooled_4ormoreppl,commute_publictransportation,commute_publictransportation_bus,commute_publictransportation_subwayorelevatedrail,commute_publictransportation_longdistancetrainorcommuterrail,commute_publictransportation_lightrailstreetcarortrolley,commute_publictransportation_ferryboat,commute_bicycle,commute_walk,commute_taxicabmotorcycleother,commute_workedfromhome,foreignborn_total,fb_europe,fb_asia,fb_africa,fb_oceania,fb_americas,fb_am_latinamerica,fb_am_northern,attainment_total_over25_series,attainment_noschooling,attainment_nurseryschool,attainment_kindergarten,attainment_1stgrade,attainment_2ndgrade,attainment_3rdgrade,attainment_4thgrade,attainment_5thgrade,attainment_6thgrade,attainment_7thgrade,attainment_8thgrade,attainment_9thgrade,attainment_10thgrade,attainment_11thgrade,attainment_12thgradenodiploma,attainment_regularhighschooldiploma,attainment_gedoralternativecredential,attainment_somecollegelessthan1year,attainment_somecollege1ormoreyearsnodegree,attainment_associatesdegree,attainment_bachelorsdegree,attainment_mastersdegree,attainment_professionalschooldegree,attainment_doctoratedegree,classworker_total_series,classworker_privateforprofit,classworker_privateforprofit_employee,classworker_privateforprofit_selfemployedincorporatedbusiness,classworker_privatenotforprofit,classworker_localgovt,classworker_stategovt,classworker_federalgovt,classworker_selfemployednonincorporatedbusiness,classworker_unpaidfamilyworker,lfstatus_total_sexbyagebyemploymentstatus16+_series,lfstatus_mciv_16to19,lfstatus_mciv_16to19_employed,lfstatus_mciv_16to19_unemployed,lfstatus_mciv_20to21,lfstatus_mciv_20to21_employed,lfstatus_mciv_20to21_unemployed,lfstatus_mciv_22to24,lfstatus_mciv_22to24_employed,lfstatus_mciv_22to24_unemployed,lfstatus_mciv_25to29,lfstatus_mciv_25to29_employed,lfstatus_mciv_25to29_unemployed,lfstatus_mciv_30to34,lfstatus_mciv_30to34_employed,lfstatus_mciv_30to34_unemployed,lfstatus_mciv_35to44,lfstatus_mciv_35to44_employed,lfstatus_mciv_35to44_unem

In [56]:
ok.tail()

,NAME,agebysex_total_series,hhsize_avg,poverty_total_bysexbyage_series,poverty_belowlevel,units_allhousing,occupancy_total_series,occupancy_occupiedunits,occupancy_vacantunits,tenure_total_series,tenure_owneroccunits,tenure_renteroccunits,units_total_series,units_one_detached,units_one_attached,units_two,units_threetofour,units_fivetonine,units_tentonineteen,units_twentytofortynine,units_fiftyormore,units_mobilehome,units_boatrvvanetc,hhtype_total_series,hhtype_familyhh,hhtype_familyhh_marriedcouplefam,hhtype_familyhh_otherfam,hhtype_familyhh_malenospouse,hhtype_familyhh_femalenospouse,hhtype_nonfamhh,hhtype_nonfamhh_householderalone,hhtype_nonfamhh_householdernotalone,hhincome_median,hhincome_total_series,hhincome_lessthan10000,hhincome_10to14999,hhincome_15to19999,hhincome_20to24999,hhincome_25to29999,hhincome_30to34999,hhincome_35to39999,hhincome_40to44999,hhincome_45to49999,hhincome_50to59999,hhincome_60to74999,hhincome_75to99999,hhincome_100to124999,hhincome125to149999,hhincome150to199999,hhincome200ormore,housingcost_total_selectedownercosts%hhincome_series,housingcost_total%ownercostwmortgage_series,housingcost_%ownercost30to34.9_wmortgage,housingcost_%ownercost35to39.9_wmortgage,housingcost_%ownercost40to49.9_wmortgage,housingcost_%ownercost50+_wmortgage,housingcost_total%ownercostwomortgage_series,housingcost_%ownercost30to34.9_womortgage,housingcost_%ownercost35to39.9_womortgage,housingcost_%ownercost40to49.9_womortgage,housingcost_%ownercost50+_womortgage,housingcost_total_rent%hhincome_series,housingcost_%rentercost30to34.9,housingcost_%rentercost35to39.9,housingcost_%rentercost40to49.9,housingcost_%rentercost50+,housingcost_medvalue_ownerocc_wmortgage,housingcost_medvalue_ownerocc_womortgage,housingcost_mediangrossrent_renteroccupied,commute_total_meansoftransportationtowork_series,commute_cartruckvan,commute_cartruckvan_drovealone,commute_cartruckvan_carpooled,commute_cartruckvan_carpooled_2ppl,commute_cartruckvan_carpooled_3ppl,commute_cartruckvan_carpooled_4ormoreppl,commute_publictransportation,commute_publictransportation_bus,commute_publictransportation_subwayorelevatedrail,commute_publictransportation_longdistancetrainorcommuterrail,commute_publictransportation_lightrailstreetcarortrolley,commute_publictransportation_ferryboat,commute_bicycle,commute_walk,commute_taxicabmotorcycleother,commute_workedfromhome,foreignborn_total,fb_europe,fb_asia,fb_africa,fb_oceania,fb_americas,fb_am_latinamerica,fb_am_northern,attainment_total_over25_series,attainment_noschooling,attainment_nurseryschool,attainment_kindergarten,attainment_1stgrade,attainment_2ndgrade,attainment_3rdgrade,attainment_4thgrade,attainment_5thgrade,attainment_6thgrade,attainment_7thgrade,attainment_8thgrade,attainment_9thgrade,attainment_10thgrade,attainment_11thgrade,attainment_12thgradenodiploma,attainment_regularhighschooldiploma,attainment_gedoralternativecredential,attainment_somecollegelessthan1year,attainment_somecollege1ormoreyearsnodegree,attainment_associatesdegree,attainment_bachelorsdegree,attainment_mastersdegree,attainment_professionalschooldegree,attainment_doctoratedegree,classworker_total_series,classworker_privateforprofit,classworker_privateforprofit_employee,classworker_privateforprofit_selfemployedincorporatedbusiness,classworker_privatenotforprofit,classworker_localgovt,classworker_stategovt,classworker_federalgovt,classworker_selfemployednonincorporatedbusiness,classworker_unpaidfamilyworker,lfstatus_total_sexbyagebyemploymentstatus16+_series,lfstatus_mciv_16to19,lfstatus_mciv_16to19_employed,lfstatus_mciv_16to19_unemployed,lfstatus_mciv_20to21,lfstatus_mciv_20to21_employed,lfstatus_mciv_20to21_unemployed,lfstatus_mciv_22to24,lfstatus_mciv_22to24_employed,lfstatus_mciv_22to24_unemployed,lfstatus_mciv_25to29,lfstatus_mciv_25to29_employed,lfstatus_mciv_25to29_unemployed,lfstatus_mciv_30to34,lfstatus_mciv_30to34_employed,lfstatus_mciv_30to34_unemployed,lfstatus_mciv_35to44,lfstatus_mciv_35to44_employed,lfstatus_mciv_35to44_unem

In [57]:
ok.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Columns: 557 entries, NAME to GeoFIPS
dtypes: float64(556), object(1)
memory usage: 352.6+ KB


In [58]:
ok.to_csv('../data/ACS20105YR.csv', index = False)